# Module 1.3 — MNIST with a Multilayer Perceptron

We'll build a fully-connected neural network from scratch using PyTorch and train it to recognise handwritten digits.

**What you'll learn:**
- How to define a network with `nn.Module`
- Activation functions: ReLU, softmax
- Cross-entropy loss and why we use it for classification
- A proper train / validation / test split
- What overfitting looks like — and how to spot it from loss curves

## 1. Setup

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print(f'PyTorch version: {torch.__version__}')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

PyTorch version: 2.2.2
Using device: mps


## 2. Load MNIST

MNIST is 70,000 grayscale 28×28 images of handwritten digits (0–9).
- 60,000 for training, 10,000 held out for test
- We'll carve 10,000 off the training set as a **validation** set

The transform converts each image to a tensor and normalises pixel values to mean=0.5, std=0.5 — so values land roughly in [-1, 1] instead of [0, 1].

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set   = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Split training → train + validation
train_set, val_set = random_split(full_train, [50000, 10000])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

print(f'Train: {len(train_set):,}  |  Val: {len(val_set):,}  |  Test: {len(test_set):,}')

Train: 50,000  |  Val: 10,000  |  Test: 10,000


### Peek at the data

In [7]:
images, labels = next(iter(train_loader))
print(f'Batch shape: {images.shape}')  # [64, 1, 28, 28]

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.tight_layout()
plt.show()

RuntimeError: Numpy is not available

## 3. Define the Network

Every PyTorch model is a class that inherits from `nn.Module`. You must implement two things:
- `__init__` — define the layers
- `forward` — describe how data flows through them

Our architecture: **784 → 256 → 128 → 10**

Each image is 28×28 = 784 pixels. We flatten it to a 1-D vector, pass it through two hidden layers with ReLU activations, and produce 10 raw scores (one per digit class).

**Why ReLU?** `ReLU(x) = max(0, x)`. It's fast, doesn't saturate for positive values, and works well in practice. Without non-linearities, stacking linear layers would collapse to a single linear transformation.

**Why no softmax in forward?** `nn.CrossEntropyLoss` applies a log-softmax internally, so we return raw logits. Adding softmax twice would give wrong gradients.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 4. Loss Function & Optimiser

**Cross-entropy loss** measures how wrong our predicted probability distribution is vs the true label. For a correct class with predicted probability p, the loss is -log(p) — so confidently wrong predictions are punished heavily.

**Adam** is an adaptive gradient descent algorithm. It tracks a running mean and variance of gradients to scale each weight's learning rate individually — usually converges faster than plain SGD.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 5. Training Loop

The core loop every deep learning training run follows:
1. **Forward pass** — run inputs through the network to get predictions
2. **Compute loss** — compare predictions to ground truth
3. **Backward pass** — compute gradients via backpropagation (`loss.backward()`)
4. **Update weights** — move each weight slightly against its gradient (`optimizer.step()`)
5. **Zero gradients** — PyTorch accumulates gradients by default, so clear them before the next batch

We run a full validation pass at the end of each epoch — **no gradient computation** (`torch.no_grad()`), which saves memory and speeds things up.

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()

    n = len(loader.dataset)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()

    n = len(loader.dataset)
    return total_loss / n, correct / n

In [ ]:
EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:02d} | '
          f'Train loss: {train_loss:.4f}, acc: {train_acc:.3f} | '
          f'Val loss: {val_loss:.4f}, acc: {val_acc:.3f}')

## 6. Visualise Training

In [ ]:
epochs = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(epochs, history['train_acc'], label='Train')
ax2.plot(epochs, history['val_acc'],   label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Test Set Evaluation

We only touch the test set **once** — at the very end. If you evaluated on it repeatedly and adjusted your model based on the result, it would effectively become part of your training process and you'd be overfitting to it without knowing.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'Test accuracy: {test_acc:.4f}  ({test_acc*100:.2f}%)')

## 8. Inspect Predictions

In [ ]:
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    logits = model(images)
    probs  = torch.softmax(logits, dim=1)
    preds  = logits.argmax(dim=1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].cpu().squeeze(), cmap='gray')
    correct = preds[i] == labels[i]
    color = 'green' if correct else 'red'
    ax.set_title(f'pred:{preds[i].item()}\ntrue:{labels[i].item()}',
                 color=color, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Overfitting Demo

Overfitting is when a model memorises the training data rather than learning general patterns — it performs well on training examples but poorly on new ones.

The classic signature: **training loss keeps falling while validation loss flattens or rises**.

We'll force it by training a much larger model on only 500 samples.

In [ ]:
from torch.utils.data import Subset

# Tiny training set — easy to overfit
tiny_train = Subset(full_train, range(500))
tiny_loader = DataLoader(tiny_train, batch_size=32, shuffle=True)

# Deliberately oversized model for 500 examples
class BigMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, 1024), nn.ReLU(),
            nn.Linear(1024, 10),
        )
    def forward(self, x):
        return self.net(x)

big_model = BigMLP().to(device)
big_opt   = optim.Adam(big_model.parameters(), lr=1e-3)

overfit_history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, 31):
    train_loss, _ = train_epoch(big_model, tiny_loader, criterion, big_opt)
    val_loss,   _ = evaluate(big_model, val_loader, criterion)
    overfit_history['train_loss'].append(train_loss)
    overfit_history['val_loss'].append(val_loss)

plt.figure(figsize=(8, 4))
plt.plot(overfit_history['train_loss'], label='Train loss')
plt.plot(overfit_history['val_loss'],   label='Val loss')
plt.title('Overfitting — large model on 500 samples')
plt.xlabel('Epoch')
plt.legend()
plt.show()
print('Notice: train loss → 0, val loss rises or plateaus high')

## Summary

| Concept | What we did |
|---|---|
| `nn.Module` | Defined our MLP as a class with `__init__` + `forward` |
| ReLU | Non-linearity that lets the network learn complex functions |
| Cross-entropy | Loss function for multi-class classification |
| Train/val/test split | Honest evaluation — test set touched exactly once |
| Overfitting | Training loss drops, val loss doesn't follow — model is memorising |

**Next up — Module 2.1:** Convolutional Neural Networks, which exploit the spatial structure of images instead of flattening everything to a vector. We'll see a significant accuracy jump on the same MNIST data.